Cześć, jest to moja perwsza styczność z tym narzędziem jak i trenowaniem sieci neuronowej. #LubięWyzwania
Niestety ale ze względu na źle oszacowany przeze mnie czas i dopinanie wyjazdu na warzną dla mnie konferencje AI
nie jestem w stanie zaprezentować aplikacji w stadium jakie ja uwarzam za poprawne i jakim bym chciał zaprezentować.
Aplikacja nie przeszła wszystkich zaplanowanych testów i nie jestem w pełni stwierdzić czy algorytm poprawnie działa.

Wykorzystane technologie zostały wybrane bazując na chęci nauczeniu się nowych rzeczy w znanym przezemnie środowisku.

IDE Visual Studio Commynity 2026

Technologie: 
- C++/Pythonb
- CMake
- Libtorch

# Struktura projektu

```
Research/
├── data/raw/                    
├── models/                   
├── notebooks/
│   └─01.ipynb 
├── results/                   
├── checkpoints/                 
├── scripts/
│   ├── exportModel.py          
│   ├── gradcam.py               
│   └── plotResults.py           
├── src/
│   ├── data/
│   │   ├── ImageDataset.h/.cpp 
│   │   ├── ImageDatasetConfig.h
│   │   └── ImageTypes.h         
│   ├── models/
│   │   ├── Classifier.h/.cpp   
│   │   └── ModelLoader.h/.cpp   
│   └── training/
│       ├── Trainer.h/.cpp      
│       └── metrics.h/.cpp       
├── vendor/
│   ├── stb_image.h              
│   └── json.hpp                
├── CMakeLists.txt
├── CMakePresets.json
└── requirements.txt
```

# Architektura rozwiązania

Projekt jest podzielony na dwie warstwy:

**Python** (`notebooks/`, `scripts/`):
- eksploracja i analiza danych,
- eksport pretrenowanych modeli do formatu TorchScript,
- wizualizacja wyników i map aktywacji.

**C++ / LibTorch** (`src/`):
- wczytywanie i preprocessing obrazów (`ImageDataset`),
- ładowanie modeli TorchScript (`ModelLoader`),
- wrapper modelu z obsługą trybów frozen/full fine-tuning (`Classifier`),
- pętla treningowa z optymalizatorem AdamW (`Trainer`),
- obliczanie metryk: F1, ROC-AUC, Precision, Recall, Confusion Matrix (`metrics`).


# Potencjalne słabości i mitygacje

Poniżej zestawiam zidentyfikowane ryzyka projektowe wraz z zastosowanymi mitygacjami.  
Każda mitygacja jest powiązana z konkretnym miejscem w kodzie lub notebooku.

| # | Problem | Opis | Mitygacja | Gdzie |
|---|---|---|---|---|
| 1 | **Niezbalansowanie klas (2:1)** | `HandDrawn` pochodzi z dwóch źródeł (`hand_photo` + `hand_scan`), `Digital` tylko z jednego (`stamp_photo`) — potencjalnie 2× więcej próbek HandDrawn | F1-score macro jako główna metryka; stratyfikowany podział po `(drawing_type × class_name)` | `metrics.cpp`, `ImageDataset::splitTrainVal()`, komórka 15 |
| 2 | **Przeuczenie przy full fine-tuning** | Pełne fine-tuning na stosunkowo małym zbiorze może nadmiernie dopasować model do danych treningowych | `weight_decay` w AdamW; checkpoint zapisywany tylko przy poprawie F1 na zbiorze **walidacyjnym** | `Trainer.cpp` |
| 3 | **Augmentacje zacierające różnice między klasami** | Zbyt agresywne augmentacje dla HandDrawn mogą sprawić, że zacznie wyglądać jak Digital i odwrotnie | Augmentacje **asymetryczne**: inne dla HandDrawn (szum, crop), inne dla Digital (skalowanie) | `ImageDataset::augment()` |
| 4 | **`conv1` pretrenowany na RGB, wejście to grayscale** | ResNet18/MobileNetV2 były trenowane na 3-kanałowych obrazach ImageNet | Wagi `conv1` uśredniane po 3 kanałach do 1 przy eksporcie — zachowuje orientację filtrów | `scripts/exportModel.py` |
| 5 | **Wyciek danych między zbiorami** | Te same obrazy w train i test zawyżają metryki testowe | Explicite sprawdzenie `Train ∩ Val = ∅`, `Train ∩ Test = ∅`, `Val ∩ Test = ∅` przez porównanie ścieżek | Komórka 16 |
| 6 | **Cechy ImageNet vs rysunki** | Backbone uczony na fotografiach — cechy mogą słabo transferować się na proste rysunki | Porównanie frozen vs full fine-tuning jako pytanie badawcze | `Trainer.cpp`, komórki 20–23 |
| 7 | **Niereprodukowane wyniki** | Brak kontroli losowości powoduje różne wyniki między uruchomieniami | Seed `42` ustawiony globalnie w C++ i Python | `Trainer.cpp`, komórka 2 |
